In [1]:
# additional Python packages required: pandas
# if you wish to install the package into the current notebook kernel, uncomment the following lines and execute this cell

# import sys
# !{sys.executable} -m pip install pandas

In [2]:
# imports

import os
import pandas as pd
import json

from typing import Dict, List, Any
from collections.abc import Callable

from dataclasses import dataclass, field

In [3]:
# helper functions, dataclass and derivation calculation

def save_to_json(export_filepath: str, data: dict):
    json_file_str: str = json.dumps(data, indent=2, sort_keys=False, ensure_ascii=False)

    if (os.path.isfile(export_filepath)):
        print(f'JSON file {export_filepath} already exists and will be overwritten.')

    with open(export_filepath, 'w') as file:
        file.write(json_file_str)
        print(f'JSON file {export_filepath} successfully created.')

def dict_prettyprint(input: dict) -> None:
    print(
        json.dumps(
            input,
            indent=2, 
            sort_keys=False, 
            ensure_ascii=False
        )
    )
    
@dataclass
class TextData:
    """Class for saving texts as data frames and derived values from the texts"""
    texts: Dict[str, pd.DataFrame] = field(default_factory=dict)
    derivations: Dict[str, List[Dict[str, Any]]] = field(default_factory=dict)
    
    def load_texts(self, filepaths: List[str], overwrite: bool = False) -> None:
        if not list(self.texts.keys()) or overwrite:
            result: dict = {}
            for filepath in filepaths:
                if not os.path.exists(filepath):
                    print(f"{filepath} does not exist. skipping file...")
                    continue
                filename = os.path.basename(filepath)
                result[filename] = pd.read_csv(filepath, encoding='utf8')
            self.texts = result
        
    def add_derivation(self, text_name: str, derivation_name: str, fn: Callable, args: dict) -> None:
        fn_result = fn(**args)
        self.derivations.setdefault(text_name, []).append({derivation_name: fn_result})

    @property
    def derivations_overview(self) -> Dict[str, Dict[str, Any]]:
        result = {}
        for text_name in self.derivations:
            _derivation_result = {}
            for derivation_dict in self.derivations[text_name]:
                derivation_name = list(derivation_dict.keys())[0]
                _derivation_result[derivation_name] = derivation_dict[derivation_name]
            result[text_name] = _derivation_result
        return result

    @property
    def text_file_list(self) -> List[str]:
        return [filename for filename in list(self.texts.keys())]

    def __repr__(self):
        result: str = ''
        for key in vars(self):
            result += key + '\n'
            for subkey in vars(self)[key]:
                result += f'  {subkey}' + '\n'
                if isinstance(vars(self)[key][subkey], list):
                    for derivation_dict in vars(self)[key][subkey]:
                        result += f'    {list(derivation_dict.keys())[0]}' + '\n'
        return result
    
def total_token_amount(input: pd.DataFrame) -> int:
    return len(input.index)

def total_cs1_main_class_annotated_token_amount(input: pd.DataFrame) -> int:
    filtered_df_all_cs1_main_annotations = input[(input['Tag'] != 'O') & (~ input['Tag'].str.contains('-ALT'))]
    return len(filtered_df_all_cs1_main_annotations.index)

def total_cs1_main_class_annotation_amount(input: pd.DataFrame) -> int:
    filtered_df_all_cs1_main_annotated_token: pd.DataFrame = input[(input['Tag'] != 'O') & (~ input['Tag'].str.contains('-ALT'))]
    filtered_df_all_cs1_main_annotations: pd.DataFrame = filtered_df_all_cs1_main_annotated_token.drop_duplicates(subset=['Annotation_ID'])
    return len(filtered_df_all_cs1_main_annotations.index)

def cs1_main_class_annotated_token_which_should_be_in_an_event(input: pd.DataFrame) -> int:
    filtered_df: pd.DataFrame = input[
        (input['main_tags_in_events'] == 'X') &
        (input['Tag'] != 'O') &
        (~ input['Tag'].str.contains('-ALT'))
    ]
    return len(filtered_df.index)

def cs1_main_class_annotated_token_which_should_be_in_a_non_event(input: pd.DataFrame) -> int:
    filtered_df: pd.DataFrame = input[
        (input['main_tags_in_non_events'] == 'X') &
        (input['Tag'] != 'O') &
        (~ input['Tag'].str.contains('-ALT'))
    ]
    return len(filtered_df.index)

def cs1_main_class_annotated_token_in_correctly_predicted_events(input: pd.DataFrame) -> int:
    filtered_df: pd.DataFrame = input[
        (input['event_label'].isin(['process', 'stative_event', 'change_of_state'])) &
        (input['main_tags_in_events'] == 'X') &
        (input['Tag'] != 'O') &
        (~ input['Tag'].str.contains('-ALT'))
    ]
    return len(filtered_df.index)

def cs1_main_class_annotated_token_in_correctly_predicted_non_events(input: pd.DataFrame) -> int:
    filtered_df: pd.DataFrame = input[
        (input['event_label'].isin(['non_event'])) &
        (input['main_tags_in_non_events'] == 'X') &
        (input['Tag'] != 'O') &
        (~ input['Tag'].str.contains('-ALT'))
    ]
    return len(filtered_df.index)

def event_classifier_accuracy_by_token(input: pd.DataFrame) -> float:
    filtered_df_all_evaluated_token: pd.DataFrame = input[input['correct_event_nonevent_prediction'] != '_']
    filtered_df_correctly_predicted_token: pd.DataFrame = filtered_df_all_evaluated_token[filtered_df_all_evaluated_token['correct_event_nonevent_prediction'] == 'X']
    return round(len(filtered_df_correctly_predicted_token.index) / len(filtered_df_all_evaluated_token.index), 3)

def cs1_main_class_annotated_token__with_diegesis_reference(input: pd.DataFrame) -> int:
    filtered_df: pd.DataFrame = input[input['reference_to_diegesis'] == 'X']
    return len(filtered_df.index)

def cs1_main_class_annotated_token_which_should_be_in_an_event__with_diegesis_reference(input: pd.DataFrame) -> int:
    filtered_df: pd.DataFrame = input[(input['reference_to_diegesis'] == 'X') & (input['main_tags_in_events'] == 'X')]
    return len(filtered_df.index)

def cs1_main_class_annotated_token_which_should_be_in_a_non_event__with_diegesis_reference(input: pd.DataFrame) -> int:
    filtered_df: pd.DataFrame = input[(input['reference_to_diegesis'] == 'X') & (input['main_tags_in_non_events'] == 'X')]
    return len(filtered_df.index)

def cs1_main_class_annotated_token_in_correctly_predicted_events__with_diegesis_reference(input: pd.DataFrame) -> int:
    filtered_df: pd.DataFrame = input[
        (input['event_label'].isin(['process', 'stative_event', 'change_of_state'])) &
        (input['main_tags_in_events'] == 'X') &
        (input['reference_to_diegesis'] == 'X') &
        (input['Tag'] != 'O') &
        (~ input['Tag'].str.contains('-ALT'))
    ]
    return len(filtered_df.index)

def cs1_main_class_annotated_token_in_correctly_predicted_non_events__with_diegesis_reference(input: pd.DataFrame) -> int:
    filtered_df: pd.DataFrame = input[
        (input['event_label'].isin(['non_event'])) &
        (input['main_tags_in_non_events'] == 'X') &
        (input['reference_to_diegesis'] == 'X') &
        (input['Tag'] != 'O') &
        (~ input['Tag'].str.contains('-ALT'))
    ]
    return len(filtered_df.index)

derivation_calculations: Dict[str, Callable] = {
    'TOTAL TOKEN AMOUNT': total_token_amount,
    'TOTAL CS1 MAIN CLASS ANNOTATED TOKEN AMOUNT': total_cs1_main_class_annotated_token_amount,
    'TOTAL CS1 MAIN CLASS ANNOTATION AMOUNT': total_cs1_main_class_annotation_amount,
    'CS1 MAIN CLASS ANNOTATED TOKEN WHICH SHOULD BE IN AN EVENT': cs1_main_class_annotated_token_which_should_be_in_an_event,
    'CS1 MAIN CLASS ANNOTATED TOKEN WHICH SHOULD BE IN A NON_EVENT': cs1_main_class_annotated_token_which_should_be_in_a_non_event,
    'CS1 MAIN CLASS ANNOTATED TOKEN IN CORRECTLY PREDICTED EVENTS': cs1_main_class_annotated_token_in_correctly_predicted_events,
    'CS1 MAIN CLASS ANNOTATED TOKEN IN CORRECTLY PREDICTED NON_EVENTS': cs1_main_class_annotated_token_in_correctly_predicted_non_events,
    'EVENT CLASSIFIER ACCURACY BY TOKEN': event_classifier_accuracy_by_token,
    'CS1 MAIN CLASS ANNOTATED TOKEN - WITH DIEGESIS REFERENCE': cs1_main_class_annotated_token__with_diegesis_reference,
    'CS1 MAIN CLASS ANNOTATED TOKEN WHICH SHOULD BE IN AN EVENT - WITH DIEGESIS REFERENCE': cs1_main_class_annotated_token_which_should_be_in_an_event__with_diegesis_reference,
    'CS1 MAIN CLASS ANNOTATED TOKEN WHICH SHOULD BE IN A NON_EVENT - WITH DIEGESIS REFERENCE': cs1_main_class_annotated_token_which_should_be_in_a_non_event__with_diegesis_reference,
    'CS1 MAIN CLASS ANNOTATED TOKEN IN CORRECTLY PREDICTED EVENTS - WITH DIEGESIS REFERENCE': cs1_main_class_annotated_token_in_correctly_predicted_events__with_diegesis_reference,
    'CS1 MAIN CLASS ANNOTATED TOKEN IN CORRECTLY PREDICTED NON_EVENTS - WITH DIEGESIS REFERENCE': cs1_main_class_annotated_token_in_correctly_predicted_non_events__with_diegesis_reference
}


In [4]:
# create /notebook_output folder

if not (os.path.isdir(os.path.join('notebook_output', 'manual_evaluation_statistics'))):
    os.makedirs(os.path.join('notebook_output', 'manual_evaluation_statistics'))

In [5]:
# execute calculations

# get filenames, create TextData instance and load texts
text_filenames: List[str] = [filename for filename in os.listdir(os.path.join('results', 'manual_evaluations')) if filename.endswith('.csv')]

text_data = TextData()
text_data.load_texts([os.path.join('results', 'manual_evaluations', filename) for filename in text_filenames])

# use the derivation calculation functions to add derivations in text_data
for text_file_name in text_data.text_file_list:
    for derivation_name in derivation_calculations:
        text_data.add_derivation(
            text_name=text_file_name,
            derivation_name=derivation_name,
            fn=derivation_calculations[derivation_name],
            args={'input': text_data.texts[text_file_name]}
        )
        
# print the calculated derivations_overview property and export it for every file
current_derivations_overview: Dict[str, Dict[str, Any]] = text_data.derivations_overview
dict_prettyprint(text_data.derivations_overview)

for text_name in current_derivations_overview:
  save_to_json(
    export_filepath=os.path.join('notebook_output', 'manual_evaluation_statistics', f'{text_name[:-4]}.json'),
    data=current_derivations_overview[text_name]
  )

{
  "DEU19_001_2-3-5.csv": {
    "TOTAL TOKEN AMOUNT": 6021,
    "TOTAL CS1 MAIN CLASS ANNOTATED TOKEN AMOUNT": 831,
    "TOTAL CS1 MAIN CLASS ANNOTATION AMOUNT": 753,
    "CS1 MAIN CLASS ANNOTATED TOKEN WHICH SHOULD BE IN AN EVENT": 700,
    "CS1 MAIN CLASS ANNOTATED TOKEN WHICH SHOULD BE IN A NON_EVENT": 131,
    "CS1 MAIN CLASS ANNOTATED TOKEN IN CORRECTLY PREDICTED EVENTS": 583,
    "CS1 MAIN CLASS ANNOTATED TOKEN IN CORRECTLY PREDICTED NON_EVENTS": 46,
    "EVENT CLASSIFIER ACCURACY BY TOKEN": 0.703,
    "CS1 MAIN CLASS ANNOTATED TOKEN - WITH DIEGESIS REFERENCE": 733,
    "CS1 MAIN CLASS ANNOTATED TOKEN WHICH SHOULD BE IN AN EVENT - WITH DIEGESIS REFERENCE": 696,
    "CS1 MAIN CLASS ANNOTATED TOKEN WHICH SHOULD BE IN A NON_EVENT - WITH DIEGESIS REFERENCE": 36,
    "CS1 MAIN CLASS ANNOTATED TOKEN IN CORRECTLY PREDICTED EVENTS - WITH DIEGESIS REFERENCE": 580,
    "CS1 MAIN CLASS ANNOTATED TOKEN IN CORRECTLY PREDICTED NON_EVENTS - WITH DIEGESIS REFERENCE": 12
  },
  "DEU19_030_1-1-1.